In [ ]:
# NAME: EFE MATTHEW AKPOVWOVWO
# STUDENT ID: x24221783

# IMPORTS
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

In [ ]:
# CONNECTION TO DATABASE & FILE PATHS
# Loading Shared Environment
load_dotenv("../data.env", override=True)

DB_USER = os.getenv("DATABASE_USERNAME")
DB_PASS = os.getenv("DATABASE_PASSWORD")
DB_HOST = os.getenv("DATABASE_HOST", "localhost")
DB_PORT = os.getenv("DATABASE_PORT", "5432")
DB_NAME = os.getenv("DATABASE_NAME")

if not all([DB_USER, DB_PASS, DB_NAME]):
    raise RuntimeError(
        "Missing required database variables. "
        "Expected: DATABASE_USERNAME, DATABASE_PASSWORD, DATABASE_NAME"
    )

print(os.getcwd())

# CSV Path
CSV_PATH = os.path.join(r"../dataset/efe_enrolment.csv")

# SQLAlchemy Connection String
CONN_STR = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# Creating SQLAlchemy Engine
def get_engine():
    engine = create_engine(CONN_STR)
    print(f"Connected to PostgreSQL at {DB_HOST}:{DB_PORT}, DB = {DB_NAME}")
    return engine

In [ ]:
print("DB_USER =", DB_USER)
print("DB_HOST =", DB_HOST)
print("DB_PORT =", DB_PORT)
print("DB_NAME =", DB_NAME)

In [ ]:
# STAGING, DIMENSION, AND FACT TABLES IN THE DATABASE SCHEMA
# Creating Core Tables In Postgresql Database
def create_tables(engine):    
    ddl_statements = [
        # Raw Staging Table
        """
        CREATE TABLE IF NOT EXISTS efe_raw_enrolment (
            id             BIGSERIAL PRIMARY KEY,
            region_country TEXT,
            year           INT,
            series         TEXT,
            value          NUMERIC,
            footnotes      TEXT,
            source         TEXT,
            raw_load_ts    TIMESTAMP DEFAULT NOW()
        );
        """,
        # Country Dimension
        """
        CREATE TABLE IF NOT EXISTS dim_country (
            country_id   SERIAL PRIMARY KEY,
            country_name TEXT NOT NULL,
            iso_alpha2   CHAR(2),
            iso_alpha3   CHAR(3),
            region       TEXT,
            income_group TEXT
        );
        """,
        # Indicator Dimension
        """
        CREATE TABLE IF NOT EXISTS dim_indicator (
            indicator_id         SERIAL PRIMARY KEY,
            original_series_name TEXT NOT NULL,
            level                TEXT,
            sex                  TEXT,
            unit                 TEXT,
            description          TEXT
        );
        """,
        # Fact Table
        """
        CREATE TABLE IF NOT EXISTS fact_enrolment (
            fact_id      BIGSERIAL PRIMARY KEY,
            country_id   INT NOT NULL REFERENCES dim_country(country_id),
            indicator_id INT NOT NULL REFERENCES dim_indicator(indicator_id),
            year         INT NOT NULL,
            value        NUMERIC,
            data_quality TEXT,
            source       TEXT,
            created_at   TIMESTAMP DEFAULT NOW()
        );
        """
    ]

    with engine.begin() as conn:
        for ddl in ddl_statements:
            conn.execute(text(ddl))

    print("Core tables have been created")




In [ ]:
# LOADING AND CLEANING OF DATASET INTO PANDAS DATAFRAME
def load_csv_to_dataframe(csv_path: str) -> pd.DataFrame:
    
    # Using The Second Row As Header
    df = pd.read_csv(csv_path, header=1)

    # Dropping Empty Rows
    df = df.dropna(how="all")

    print("Original CSV columns:", df.columns.tolist())
    print("First 5 rows from CSV:")
    print(df.head())

    # Creating A region_country Label For The Unnamed Column
    if "Unnamed: 1" in df.columns:
        df["region_country"] = df["Unnamed: 1"].astype(str).str.strip()
    elif "Region/Country/Area" in df.columns:
        # Fallback, if column naming changes
        df["region_country"] = df["Region/Country/Area"].astype(str).str.strip()
    else:
        raise ValueError("Could not locate region/country column in CSV.")

    # Mapping UN Column Names To My Internal Names
    rename_map = {}
    if "Year" in df.columns:
        rename_map["Year"] = "year"
    if "Series" in df.columns:
        rename_map["Series"] = "series"
    if "Value" in df.columns:
        rename_map["Value"] = "value"
    if "Footnotes" in df.columns:
        rename_map["Footnotes"] = "footnotes"
    if "Source" in df.columns:
        rename_map["Source"] = "source"

    df = df.rename(columns=rename_map)

    needed = ["region_country", "year", "series", "value", "footnotes", "source"]
    df = df[needed]

    # Basic Cleaning
    df["region_country"] = df["region_country"].astype(str).str.strip()
    df["series"] = df["series"].astype(str).str.strip()
    df["year"] = df["year"].astype(int)
    df["value"] = pd.to_numeric(
        df["value"].replace({",": ""}, regex=True),
        errors="coerce",
    )

    print("Cleaned DataFrame:")
    print(df.head())

    return df

In [ ]:
# LOADING RAW DATASET INTO POSTGRESSQL STAGING TABLE
def load_raw_to_postgres(df_raw: pd.DataFrame, engine):    
    with engine.begin() as conn:
        conn.execute(text("TRUNCATE TABLE efe_raw_enrolment RESTART IDENTITY;"))

    df_raw.to_sql("efe_raw_enrolment", engine, if_exists="append", index=False)
    print(f"Inserted {len(df_raw)} rows into efe_raw_enrolment.")


In [ ]:
# BUILDING DIMENSIONAL LOOKUP TABLES

def parse_series(series_name: str):
    s = str(series_name).lower()

    level = None
    sex = "total"
    unit = None

    # Education Level
    if "primary" in s:
        level = "primary"
    elif "lower secondary" in s:
        level = "lower_secondary"
    elif "upper secondary" in s:
        level = "upper_secondary"

    # Sex
    if "female" in s:
        sex = "female"
    elif "male" in s:
        sex = "male"
    else:
        sex = "total"

    # Units
    if "thousands" in s:
        unit = "thousands"
    elif "ratio" in s or "%" in s or "percent" in s:
        unit = "percent"

    return level, sex, unit


# Quick test for parse_series
test_series = [
    "Gross enrollment ratio - Primary (female)",
    "Gross enrollment ratio - Primary (male)",
    "Students enrolled in primary education (thousands)"
]

for item in test_series:
    print(item, "->", parse_series(item))


# Building dim_country From region_country Label Created Earlier

def build_dim_country(df_raw: pd.DataFrame, engine) -> pd.DataFrame:
    countries = (
        df_raw[["region_country"]]
        .drop_duplicates()
        .rename(columns={"region_country": "country_name"})
    )

    # Creating iso_alpha3 Column With NGA Country Code for Nigeria
    countries["iso_alpha3"] = np.where(
        countries["country_name"] == "Nigeria",
        "NGA",
        None
    )
    countries["iso_alpha2"] = None
    countries["region"] = None
    countries["income_group"] = None

    # Using CASCADE So We Can Truncate Even If fact_enrolment References This Table
    with engine.begin() as conn:
        conn.execute(
            text("TRUNCATE TABLE dim_country RESTART IDENTITY CASCADE;")
        )

    countries.to_sql(
        "dim_country",
        engine,
        if_exists="append",
        index=False
    )

    dim_country = pd.read_sql(
        "SELECT country_id, country_name, iso_alpha3 FROM dim_country",
        engine
    )

    print(f"dim_country rows: {len(dim_country)}")
    print(dim_country.head())

    return dim_country


# Building dim_indicator From Series String

def build_dim_indicator(df_raw: pd.DataFrame, engine) -> pd.DataFrame:
    indicators = (
        df_raw[["series"]]
        .drop_duplicates()
        .rename(columns={"series": "original_series_name"})
    )

    # Applying parse_series
    parsed = indicators["original_series_name"].apply(parse_series)

    indicators["level"] = parsed.apply(lambda x: x[0])
    indicators["sex"] = parsed.apply(lambda x: x[1])
    indicators["unit"] = parsed.apply(lambda x: x[2])
    indicators["description"] = indicators["original_series_name"]

    # Using CASCADE For Safety Incase More Fact Tables Reference dim_indicator Later
    with engine.begin() as conn:
        conn.execute(
            text("TRUNCATE TABLE dim_indicator RESTART IDENTITY CASCADE;")
        )

    indicators.to_sql(
        "dim_indicator",
        engine,
        if_exists="append",
        index=False
    )

    dim_indicator = pd.read_sql(
        """
        SELECT
            indicator_id,
            original_series_name,
            level,
            sex,
            unit
        FROM dim_indicator
        """,
        engine
    )

    print(f"dim_indicator rows: {len(dim_indicator)}")
    print(dim_indicator.head())

    return dim_indicator

In [ ]:
# BULDING AND POPULATING THE FACT ENROLMENT TABLE
def build_fact_enrolment(
    df_raw: pd.DataFrame,
    dim_country: pd.DataFrame,
    dim_indicator: pd.DataFrame,
    engine,
):
    # Merging Raw Data To dim_country and dim_indicator
    merged = df_raw.merge(
        dim_country,
        left_on="region_country",
        right_on="country_name",
        how="left",
    )

    merged = merged.merge(
        dim_indicator,
        left_on="series",
        right_on="original_series_name",
        how="left",
    )

    # Data Quality Flag
    merged["data_quality"] = merged["value"].apply(
        lambda v: "missing" if pd.isna(v) else "original"
    )

    fact = merged[["country_id", "indicator_id", "year", "value", "source", "data_quality"]]

    # Truncating Fact Table Before Reloading
    with engine.begin() as conn:
        conn.execute(text("TRUNCATE TABLE fact_enrolment RESTART IDENTITY;"))

    fact.to_sql("fact_enrolment", engine, if_exists="append", index=False)

    print(f"Inserted {len(fact)} rows into fact_enrolment")

In [ ]:
# CREATING NIGERIA VIEW AND WIDE TABLES 
# Nigeria View Table
def create_un_nigeria_view(engine):    
    sql = """
    CREATE OR REPLACE VIEW un_nigeria_enrolment AS
    SELECT
        c.country_name,
        COALESCE(c.iso_alpha3, 'NGA') AS country_code,
        f.year,
        i.original_series_name,
        i.level,
        i.sex,
        f.value AS enrolment_value,
        f.data_quality,
        f.source
    FROM fact_enrolment f
    JOIN dim_country   c ON f.country_id   = c.country_id
    JOIN dim_indicator i ON f.indicator_id = i.indicator_id
    WHERE c.country_name = 'Nigeria';
    """
    with engine.begin() as conn:
        conn.execute(text(sql))
    print("Created/updated view: un_nigeria_enrolment")

# Nigeria Wide Table
def create_un_nigeria_wide_table(engine):    
    drop_sql = "DROP TABLE IF EXISTS un_nigeria_wide;"
    create_sql = """
    CREATE TABLE un_nigeria_wide AS
    SELECT
        COALESCE(c.iso_alpha3, 'NGA') AS country_code,
        f.year,
        MAX(CASE WHEN i.level = 'primary'
                  AND i.sex = 'male'   THEN f.value END) AS un_primary_male,
        MAX(CASE WHEN i.level = 'primary'
                  AND i.sex = 'female' THEN f.value END) AS un_primary_female,
        MAX(CASE WHEN i.level = 'lower_secondary'
                  AND i.sex = 'male'   THEN f.value END) AS un_lower_secondary_male,
        MAX(CASE WHEN i.level = 'lower_secondary'
                  AND i.sex = 'female' THEN f.value END) AS un_lower_secondary_female,
        MAX(CASE WHEN i.level = 'upper_secondary'
                  AND i.sex = 'male'   THEN f.value END) AS un_upper_secondary_male,
        MAX(CASE WHEN i.level = 'upper_secondary'
                  AND i.sex = 'female' THEN f.value END) AS un_upper_secondary_female
    FROM fact_enrolment f
    JOIN dim_country   c ON f.country_id   = c.country_id
    JOIN dim_indicator i ON f.indicator_id = i.indicator_id
    WHERE c.country_name = 'Nigeria'
    GROUP BY COALESCE(c.iso_alpha3, 'NGA'), f.year
    ORDER BY f.year;
    """

    with engine.begin() as conn:
        conn.execute(text(drop_sql))
        conn.execute(text(create_sql))

    print("Created table: un_nigeria_wide")

In [ ]:
# ROW COUNT & SAMPLE CHECKS
# Printing Number Of Rows In Each Core Table
def print_row_counts(engine):
    tables = ["efe_raw_enrolment", "dim_country", "dim_indicator", "fact_enrolment"]
    with engine.begin() as conn:
        for tbl in tables:
            result = conn.execute(text(f"SELECT COUNT(*) FROM {tbl};"))
            count = result.scalar()
            print(f"{tbl}: {count} rows")

# Showing A Few Joined Rows From fact_enrolment
def print_sample_fact_rows(engine, limit: int = 10):
    sample_sql = f"""
        SELECT
            f.fact_id,
            c.country_name,
            f.year,
            i.level,
            i.sex,
            f.value,
            f.data_quality
        FROM fact_enrolment f
        JOIN dim_country   c ON f.country_id = c.country_id
        JOIN dim_indicator i ON f.indicator_id = i.indicator_id
        LIMIT {limit};
    """
    sample_df = pd.read_sql(sample_sql, engine)
    print("Sample rows from joined fact_enrolment and dims:")
    print(sample_df)


In [ ]:
# READING SQL SAFELY
# Using Raw Connection Instead Of SQLAlchemy Connection To Avoid Issues With Newer SQLAlchemy Versions
def read_sql_df(engine, query: str) -> pd.DataFrame:    
    raw_conn = engine.raw_connection()
    try:
        df = pd.read_sql(query, raw_conn)
    finally:
        raw_conn.close()
    return df

In [ ]:
# VISUALISATIONS
# Generating Visuals
def generate_visualisations(engine):
    viz_dir = os.path.join("..", "visualisations")
    os.makedirs(viz_dir, exist_ok=True)

    # Primary enrolment ratio by sex
    primary_query = """
        SELECT
            f.year,
            i.sex,
            f.value
        FROM fact_enrolment f
        JOIN dim_country c
            ON f.country_id = c.country_id
        JOIN dim_indicator i
            ON f.indicator_id = i.indicator_id
        WHERE c.country_name = 'Nigeria'
          AND i.level = 'primary'
          AND i.unit = 'percent'
          AND i.sex IN ('male', 'female')
        ORDER BY f.year, i.sex;
    """

    primary_df = read_sql_df(engine, primary_query)

    if not primary_df.empty:
        plt.figure(figsize=(9, 5))

        sns.lineplot(
            data=primary_df,
            x="year",
            y="value",
            hue="sex",
            marker="o"
        )

        plt.title("Nigeria - UN Primary Gross Enrolment Ratio by Sex")
        plt.xlabel("Year")
        plt.ylabel("Gross Enrolment Ratio (%)")
        plt.grid(True, linestyle="--", alpha=0.5)
        plt.tight_layout()

        file_path = os.path.join(
            viz_dir,
            "efe_un_primary_enrolment_by_sex.png"
        )

        plt.savefig(file_path, dpi=300)
        plt.show()

        print(f"Saved: {file_path}")


    # Average enrolment ratio by level and sex
    level_query = """
        SELECT
            i.level,
            i.sex,
            AVG(f.value) AS avg_value
        FROM fact_enrolment f
        JOIN dim_country c
            ON f.country_id = c.country_id
        JOIN dim_indicator i
            ON f.indicator_id = i.indicator_id
        WHERE c.country_name = 'Nigeria'
          AND i.unit = 'percent'
          AND i.sex IN ('male', 'female')
        GROUP BY i.level, i.sex
        ORDER BY i.level, i.sex;
    """

    level_df = read_sql_df(engine, level_query)

    if not level_df.empty:
        plt.figure(figsize=(9, 5))

        sns.barplot(
            data=level_df,
            x="level",
            y="avg_value",
            hue="sex"
        )

        plt.title("Nigeria - Average UN Enrolment Ratio by Level and Sex")
        plt.xlabel("Education Level")
        plt.ylabel("Average Gross Enrolment Ratio (%)")
        plt.tight_layout()

        file_path = os.path.join(
            viz_dir,
            "efe_un_enrolment_by_level_and_sex.png"
        )

        plt.savefig(file_path, dpi=300)
        plt.show()

        print(f"Saved: {file_path}")


    # Total students enrolled
    total_query = """
        SELECT
            f.year,
            i.level,
            f.value
        FROM fact_enrolment f
        JOIN dim_country c
            ON f.country_id = c.country_id
        JOIN dim_indicator i
            ON f.indicator_id = i.indicator_id
        WHERE c.country_name = 'Nigeria'
          AND i.unit = 'thousands'
          AND i.sex = 'total'
        ORDER BY f.year, i.level;
    """

    total_df = read_sql_df(engine, total_query)

    if not total_df.empty:
        plt.figure(figsize=(9, 5))

        sns.lineplot(
            data=total_df,
            x="year",
            y="value",
            hue="level",
            marker="o"
        )

        plt.title("Nigeria - UN Student Enrolment by Education Level")
        plt.xlabel("Year")
        plt.ylabel("Students Enrolled (Thousands)")
        plt.grid(True, linestyle="--", alpha=0.5)
        plt.tight_layout()

        file_path = os.path.join(
            viz_dir,
            "efe_un_total_enrolment_by_level.png"
        )

        plt.savefig(file_path, dpi=300)
        plt.show()

        print(f"Saved: {file_path}")

In [ ]:
# MANAGING THE ENTIRE PROCESS
def main():
    engine = get_engine()
    create_tables(engine)

    # Loading And Cleaning CSV Into DataFrame
    df_raw = load_csv_to_dataframe(CSV_PATH)

    # Loading Raw Into PostgreSQL Staging Table
    load_raw_to_postgres(df_raw, engine)

    # Building And Loading Dimension Tables
    dim_country = build_dim_country(df_raw, engine)
    dim_indicator = build_dim_indicator(df_raw, engine)

    # Building And Loading Fact Table
    build_fact_enrolment(df_raw, dim_country, dim_indicator, engine)

    # Nigeria View and Wide Tables For Integration
    create_un_nigeria_view(engine)
    create_un_nigeria_wide_table(engine)

    # Quick Sanity Checks
    print_row_counts(engine)
    print_sample_fact_rows(engine, limit=10)

    # Generate Efe's Visualizations
    print("Generating Efe's visualizations...")
    generate_visualisations(engine)

    print("Efe pipeline and visualizations complete.")


if __name__ == "__main__":
    main()

In [ ]:
engine = get_engine()

indicator_check = read_sql_df(
    engine,
    """
    SELECT
        indicator_id,
        original_series_name,
        level,
        sex,
        unit
    FROM dim_indicator
    ORDER BY level, sex;
    """
)

display(indicator_check)

In [ ]:
nigeria_wide_check = read_sql_df(
    engine,
    """
    SELECT *
    FROM un_nigeria_wide
    ORDER BY year;
    """
)

display(nigeria_wide_check)